# VLM Recognition Engine 

Two-round multi-VLM classification pipeline for quasar flare candidates.

- **Classifier A** (high recall): Grok 4.1 Fast
- **Classifier B** (high precision): Qwen 3.5 Plus
- **Evaluator**: (high accuracy): GPT-5

**Round 1**: Both classifiers examine all candidates independently.
Agreements are accepted as final. The evaluator reviews disagreements
and generates feedback.

**Round 2**: Classifiers re-examine only the disputed cases with
evaluator feedback injected. If they now agree, that label is final.
Remaining disagreements go to the evaluator for a final ruling.

In [1]:
import os
import json
import base64
import shutil
import time
import logging
from pathlib import Path
from datetime import datetime
from concurrent.futures import ThreadPoolExecutor, as_completed
import threading

import pandas as pd
from tqdm.notebook import tqdm
from dotenv import load_dotenv
from openai import OpenAI

for _name in ["httpx", "openai", "httpcore", "urllib3"]:
    logging.getLogger(_name).setLevel(logging.CRITICAL)

In [ ]:
# ================================================================
# CONFIGURATION
# ================================================================

PLOT_DIR       = Path(r'path')
CANDIDATE_CSV  = Path(r'path')
OUTPUT_DIR     = Path(r'path')
FLARE_DIR      = Path(r'path')


for d in [OUTPUT_DIR, FLARE_DIR]:
    d.mkdir(parents=True, exist_ok=True)

MAX_WORKERS  = 3
MAX_RETRIES  = 3
RETRY_DELAY  = 5

CLASSIFIER_A_MODEL = "x-ai/grok-4.1-fast"
CLASSIFIER_B_MODEL = "qwen/qwen3.5-plus-02-15"
EVALUATOR_MODEL    = "openai/gpt-5"

load_dotenv()
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

print(f"Plots: {PLOT_DIR}")
print(f"API key loaded: {'Yes' if OPENROUTER_API_KEY else 'NO'}")

In [3]:
# ================================================================
# HELPERS
# ================================================================

def encode_image(path: str) -> str:
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")


def call_vlm(model: str, system_prompt: str, user_text: str,
             image_b64: str | None = None) -> str:
    if image_b64 is not None:
        user_content = [
            {"type": "text", "text": user_text},
            {"type": "image_url",
             "image_url": {"url": f"data:image/png;base64,{image_b64}"}},
        ]
    else:
        user_content = user_text

    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_content},
                ],
                max_tokens=512,
            )
            return resp.choices[0].message.content.strip()
        except Exception:
            if attempt < MAX_RETRIES:
                time.sleep(RETRY_DELAY * attempt)
    return "ERROR: all attempts failed"


def parse_classification(text: str) -> str:
    t = text.lower()
    if "non_flare" in t or "non-flare" in t or "no flare" in t:
        return "non_flare"
    if "flare" in t:
        return "flare"
    return "uncertain"

In [4]:
# ================================================================
# PROMPTS
# ================================================================

CLASSIFIER_SYSTEM_PROMPT = """You are an expert astronomer classifying quasar light curves for transient flares.

You will receive an r-band light-curve image (magnitude vs MJD).
IMPORTANT: The magnitude axis is inverted — lower magnitude = higher brightness.
A brightening event appears as points moving UPWARD on the plot.

Your task is to identify if the light curve has flaring activity in it. Flaring activity means there is decrease in
magnitude and you will be able to figure out a wave like structure against the noise/damped random walk nature of the 
quasar light curves. Usual examples of flare types are gamma, gaussian, fast rise exponential decay, but these are not the only types of flare. The botttomline is that there will a rise and then decay.
Single point spikes are artifacts and donot classify them as flares. The magnitude usually decreases
by around ~0.5 (0.4 or 0.35 also works). This decrease in magnitude is from the start of the flare to the peak. It may happen 
that the peak is towards the end or towards the beginning of the light curves. You will thus see only the rising or the falling part of the flaring activity.
Such cases are also flares, and the pattern may strech from start to end or vice versa meaning there can be maximum intensity at the beginning or end and the magnitude 
keeps on increasing from there. Again remember that the y axis is inverted, since in astronomy less magnitude means higher brightness.


Respond ONLY in this format:
classification: <flare or non_flare>
reasoning: <Short reason for your classification>
"""

CLASSIFIER_USER_PROMPT = "Classify this quasar light curve: flare or non_flare?"


def build_classifier_prompt_with_feedback(feedback: str) -> str:
    return (
        f"{CLASSIFIER_SYSTEM_PROMPT}\n\n"
        f"EVALUATOR FEEDBACK FROM PREVIOUS ROUND:\n{feedback}\n\n"
        f"Use this feedback to correct your classifications."
    )


EVALUATOR_SYSTEM_PROMPT = """You are a senior astrophysicist evaluating a quasar flare detection pipeline.
Two classifiers have independently examined each light curve and DISAGREE.

For the light curve under review:
- The magnitude axis is inverted (lower mag = brighter).
- Single-point spikes are NOT flares.

Your task is to review the light curve and then the classifiers responses. Flaring activity means there is decrease in
magnitude and you will be able to figure out a wave like structure against the noise/damped random walk nature of the 
quasar light curves. Usual examples of flare types are gamma, gaussian, fast rise exponential decay, but these are not the only types of flare. The botttomline is that there will a rise and then decay.
Single point spikes are artifacts and donot classify them as flares. The magnitude usually decreases
by around ~0.5 (0.4 or 0.35 also works). This decrease in magnitude is from the start of the flare to the peak. It may happen 
that the peak is towards the end or towards the beginning of the light curves. You will thus see only the rising or the falling part of the flaring activity.
Such cases are also flares, and the pattern may strech from start to end or vice versa meaning there can be maximum intensity at the beginning or end and the magnitude 
keeps on increasing from there. Again remember that the y axis is inverted, since in astronomy less magnitude means higher brightness.


Review the image and both classifier outputs. Respond in this format:
final_classification: <flare or non_flare>
reasoning: <brief explanation>
"""

FEEDBACK_REQUEST_PROMPT = (
    "Based on the evaluations above, provide concise actionable feedback "
    "for the classifiers to reduce errors in the next round. Focus on mistake patterns."
)

In [5]:
# ================================================================
# CLASSIFICATION (parallel)
# ================================================================

def classify_single(image_path: Path, system_prompt: str, model: str) -> dict:
    try:
        img_b64  = encode_image(str(image_path))
        response = call_vlm(model, system_prompt, CLASSIFIER_USER_PROMPT, img_b64)
        label    = parse_classification(response)
    except Exception as exc:
        response = f"ERROR: {exc}"
        label    = "error"
    return {"filename": image_path.name, "label": label, "raw_response": response, "model": model}


def run_classifiers(image_paths: list[Path], system_prompt: str) -> tuple[list, list]:
    results_a, results_b = [], []
    lock = threading.Lock()

    def _classify_both(img_path):
        ra = classify_single(img_path, system_prompt, CLASSIFIER_A_MODEL)
        rb = classify_single(img_path, system_prompt, CLASSIFIER_B_MODEL)
        return ra, rb

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(_classify_both, p): p for p in image_paths}
        for future in tqdm(as_completed(futures), total=len(futures), desc="Classifying"):
            ra, rb = future.result()
            with lock:
                results_a.append(ra)
                results_b.append(rb)
    return results_a, results_b

In [6]:
# ================================================================
# EVALUATOR (parallel, disagreements only)
# ================================================================

def _evaluate_single(item: dict, img_map: dict) -> dict:
    fn = item["filename"]
    img_path = img_map.get(fn)
    if img_path is None:
        return {"filename": fn, "evaluator_label": "error", "evaluator_response": "image not found"}

    eval_prompt = (
        f"Light curve: {fn}\n"
        f"Classifier A (high recall): {item['classifier_a_label']}\n"
        f"  {item['classifier_a_response']}\n\n"
        f"Classifier B (high precision): {item['classifier_b_label']}\n"
        f"  {item['classifier_b_response']}\n\n"
        f"These classifiers disagree. Review the image and give your final classification."
    )
    try:
        img_b64  = encode_image(str(img_path))
        response = call_vlm(EVALUATOR_MODEL, EVALUATOR_SYSTEM_PROMPT, eval_prompt, img_b64)
        label    = parse_classification(response)
    except Exception as exc:
        response = f"ERROR: {exc}"
        label    = "error"
    return {"filename": fn, "evaluator_label": label, "evaluator_response": response}


def run_evaluator(disagreements: list, image_paths: list[Path]) -> tuple[list, str]:
    if not disagreements:
        return [], "No disagreements to review."

    img_map = {p.name: p for p in image_paths}
    evaluated = []
    lock = threading.Lock()

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(_evaluate_single, item, img_map): item for item in disagreements}
        for future in tqdm(as_completed(futures), total=len(futures), desc="Evaluating"):
            with lock:
                evaluated.append(future.result())

    # Generate feedback for classifiers
    summary = "\n".join(
        f"- {er['filename']}: {er['evaluator_label']}. {er['evaluator_response'][:200]}"
        for er in evaluated
    )
    feedback = call_vlm(
        EVALUATOR_MODEL, EVALUATOR_SYSTEM_PROMPT,
        f"Evaluator verdicts on disputed cases:\n\n{summary}\n\n{FEEDBACK_REQUEST_PROMPT}",
    )
    return evaluated, feedback

In [ ]:
# ================================================================
# ROUND 1 — Classify all, evaluator resolves disagreements
# ================================================================

image_paths = sorted(PLOT_DIR.glob("*.png"))
print(f"Candidates: {len(image_paths)}")
print(f"\n{'='*50}")
print("ROUND 1 — Initial classification")
print(f"{'='*50}")

# Classify all
r1_a, r1_b = run_classifiers(image_paths, CLASSIFIER_SYSTEM_PROMPT)

# Merge
map_a = {r["filename"]: r for r in r1_a}
map_b = {r["filename"]: r for r in r1_b}

agreed, disagreed = [], []
for fn in sorted(set(map_a) | set(map_b)):
    ra = map_a.get(fn, {})
    rb = map_b.get(fn, {})
    la = ra.get("label", "missing")
    lb = rb.get("label", "missing")
    rec = {
        "filename": fn,
        "classifier_a_label": la, "classifier_a_response": ra.get("raw_response", ""),
        "classifier_b_label": lb, "classifier_b_response": rb.get("raw_response", ""),
    }
    if la == lb and la not in ("error", "uncertain"):
        agreed.append(rec)
    else:
        disagreed.append(rec)

print(f"Agreed: {len(agreed)} | Disagreed: {len(disagreed)}")

# Evaluator resolves disagreements
r1_eval, r1_feedback = run_evaluator(disagreed, image_paths)

# Build Round 1 final labels
r1_eval_map = {r["filename"]: r for r in r1_eval}
r1_labels = {}

for rec in agreed:
    fn = rec["filename"]
    r1_labels[fn] = {
        "filename": fn,
        "dbID": int(fn.replace(".png", "")),
        "final_label": rec["classifier_a_label"],
        "source": "classifier_agreement",
        "classifier_a": rec["classifier_a_label"],
        "classifier_b": rec["classifier_b_label"],
        "evaluator": "not_needed",
    }

for rec in disagreed:
    fn = rec["filename"]
    ev = r1_eval_map.get(fn, {})
    ev_label = ev.get("evaluator_label", "uncertain")
    r1_labels[fn] = {
        "filename": fn,
        "dbID": int(fn.replace(".png", "")),
        "final_label": ev_label if ev_label not in ("error", "uncertain") else "uncertain",
        "source": "evaluator_round1",
        "classifier_a": rec["classifier_a_label"],
        "classifier_b": rec["classifier_b_label"],
        "evaluator": ev_label,
    }

r1_final = list(r1_labels.values())
n_f = sum(1 for r in r1_final if r["final_label"] == "flare")
n_n = sum(1 for r in r1_final if r["final_label"] == "non_flare")
n_u = sum(1 for r in r1_final if r["final_label"] == "uncertain")
print(f"\nRound 1 results — Flares: {n_f} | Non-flares: {n_n} | Uncertain: {n_u}")

pd.DataFrame(r1_final).to_csv(OUTPUT_DIR / "round1_labels.csv", index=False)

In [ ]:
# ================================================================
# ROUND 2 — Reclassify disputed cases with feedback
# ================================================================

# Only re-examine cases that were disagreements in Round 1
disputed_files = [fn for fn in [r["filename"] for r in disagreed]]
disputed_paths = [p for p in image_paths if p.name in disputed_files]

if not disputed_paths:
    print("No disagreements — skipping Round 2")
    final_labels = r1_final
else:
    print(f"\n{'='*50}")
    print(f"ROUND 2 — Reclassifying {len(disputed_paths)} disputed cases with feedback")
    print(f"{'='*50}")

    r2_prompt = build_classifier_prompt_with_feedback(r1_feedback)
    r2_a, r2_b = run_classifiers(disputed_paths, r2_prompt)

    # Merge Round 2 results
    r2_map_a = {r["filename"]: r for r in r2_a}
    r2_map_b = {r["filename"]: r for r in r2_b}

    r2_agreed, r2_disagreed = [], []
    for fn in disputed_files:
        ra = r2_map_a.get(fn, {})
        rb = r2_map_b.get(fn, {})
        la = ra.get("label", "missing")
        lb = rb.get("label", "missing")
        rec = {
            "filename": fn,
            "classifier_a_label": la, "classifier_a_response": ra.get("raw_response", ""),
            "classifier_b_label": lb, "classifier_b_response": rb.get("raw_response", ""),
        }
        if la == lb and la not in ("error", "uncertain"):
            r2_agreed.append(rec)
        else:
            r2_disagreed.append(rec)

    print(f"Now agreed: {len(r2_agreed)} | Still disagreed: {len(r2_disagreed)}")

    # Evaluator makes final call on remaining disagreements
    r2_eval, _ = run_evaluator(r2_disagreed, image_paths)
    r2_eval_map = {r["filename"]: r for r in r2_eval}

    # Build final labels: start from Round 1 agreed, add Round 2 results
    final_labels = [r for r in r1_final if r["source"] == "classifier_agreement"]

    for rec in r2_agreed:
        fn = rec["filename"]
        final_labels.append({
            "filename": fn,
            "dbID": int(fn.replace(".png", "")),
            "final_label": rec["classifier_a_label"],
            "source": "classifier_agreement_round2",
            "classifier_a": rec["classifier_a_label"],
            "classifier_b": rec["classifier_b_label"],
            "evaluator": "not_needed",
        })

    for rec in r2_disagreed:
        fn = rec["filename"]
        ev = r2_eval_map.get(fn, {})
        ev_label = ev.get("evaluator_label", "uncertain")
        final_labels.append({
            "filename": fn,
            "dbID": int(fn.replace(".png", "")),
            "final_label": ev_label if ev_label not in ("error", "uncertain") else "uncertain",
            "source": "evaluator_round2",
            "classifier_a": rec["classifier_a_label"],
            "classifier_b": rec["classifier_b_label"],
            "evaluator": ev_label,
        })

    n_f = sum(1 for r in final_labels if r["final_label"] == "flare")
    n_n = sum(1 for r in final_labels if r["final_label"] == "non_flare")
    n_u = sum(1 for r in final_labels if r["final_label"] == "uncertain")
    print(f"\nFinal results — Flares: {n_f} | Non-flares: {n_n} | Uncertain: {n_u}")

    pd.DataFrame(final_labels).to_csv(OUTPUT_DIR / "round2_labels.csv", index=False)

In [ ]:
# ================================================================
# SAVE CONFIRMED FLARES
# ================================================================

confirmed = [r for r in final_labels if r["final_label"] == "flare"]
pd.DataFrame(confirmed).to_csv(OUTPUT_DIR / "confirmed_flares.csv", index=False)

for r in confirmed:
    src = PLOT_DIR / r["filename"]
    if src.exists():
        shutil.copy2(src, FLARE_DIR / r["filename"])

print(f"Confirmed flares: {len(confirmed)}")
for r in sorted(confirmed, key=lambda x: x["dbID"]):
    print(f"  {r['dbID']}")

In [ ]:
# ================================================================
# EXPORT
# ================================================================

final_df = pd.DataFrame(final_labels)
catalog  = pd.read_csv(CANDIDATE_CSV)
results  = final_df.merge(catalog, on='dbID', how='left')
results.to_csv(OUTPUT_DIR / "vlm_results_full.csv", index=False)

# Summary
n_agreed_r1      = len(agreed)
n_disagreed_r1   = len(disagreed)
n_resolved_r2    = len(r2_agreed) if disputed_paths else 0
n_evaluator_r2   = len(r2_disagreed) if disputed_paths else 0

run_metadata = {
    "pipeline": "FLARE v2 — Iterative DRW + EVT + VLM",
    "timestamp": datetime.now().isoformat(),
    "n_candidates": len(image_paths),
    "models": {
        "classifier_a": CLASSIFIER_A_MODEL,
        "classifier_b": CLASSIFIER_B_MODEL,
        "evaluator": EVALUATOR_MODEL,
    },
    "round1": {
        "agreed": n_agreed_r1,
        "disagreed": n_disagreed_r1,
    },
    "round2": {
        "resolved_by_agreement": n_resolved_r2,
        "resolved_by_evaluator": n_evaluator_r2,
    },
    "confirmed_flares": len(confirmed),
    "confirmed_dbIDs": sorted([r["dbID"] for r in confirmed]),
}

with open(OUTPUT_DIR / "vlm_run_metadata.json", "w") as f:
    json.dump(run_metadata, f, indent=2)

print(f"\nResults: {OUTPUT_DIR / 'vlm_results_full.csv'}")
print(f"Metadata: {OUTPUT_DIR / 'vlm_run_metadata.json'}")